# Wrist activity classifier: PAMAP2 pipeline (sedentary / walking / running)
Kenneth Dialogo, 247-519-VA Project I

**What this notebook does**
1. Loads the 9 PAMAP2 wrist files (`subject101.csv.gz` ... `subject109.csv.gz`), built from the raw
   protocol **and optional** sessions, dominant-wrist sensor only.
2. Converts them to match the watch: **g** and **deg/s**, **40 Hz**.
3. Cuts 1.5 s windows (60 samples) and computes orientation-free features.
4. Trains a random forest, a shallow decision tree and a logistic-regression baseline.
5. Tests with **leave-one-subject-out**: train on 8 people, test on the 9th, repeat 9 times.
6. Shows what the model says during activities it was **never trained on** (stairs, chores).
7. Applies the same smoothing as the watch (label changes only after 3 agreeing windows).

**Classes.** Sedentary = lying, sitting, standing, watching TV, computer work, car driving.
Walking = walking. Running = running. Rows labelled `test_only` (stairs, ironing, folding laundry,
vacuuming, house cleaning) are never used for training.

**Before running:** upload the 9 `.csv.gz` files with the folder icon on the left (they land in
`/content`; no need to unzip them). Then choose **Runtime > Run all**.

This is wrist-only public data. The fusion claim (C9) still needs your own thigh-board recordings.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample_poly
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

DATA_DIR = "/content"          # change if your files are somewhere else
FS_IN, FS_OUT = 100, 40        # PAMAP2 rate -> watch recorder rate
WIN = 60                       # 1.5 s at 40 Hz (same as the reference paper)
G = 9.80665                    # m/s^2 per g
CLASSES = ["sedentary", "walking", "running"]

files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*subject1*.csv*"), recursive=True))
print(len(files), "files found")
assert files, "No subject CSVs found. Upload them or change DATA_DIR."

## 1. Load and convert to the watch's units and rate
PAMAP2 has gaps between activities, and each subject can have two sessions (protocol and optional),
so each unbroken stretch of one activity is resampled on its own (otherwise the resampler would
smear two activities together).

In [ ]:
def load_subject(path):
    d = pd.read_csv(path)
    d[["acc_x", "acc_y", "acc_z"]] /= G                 # m/s^2 -> g
    d[["gyro_x", "gyro_y", "gyro_z"]] *= 180 / np.pi    # rad/s -> deg/s
    # new segment whenever the activity changes or there is a gap longer than 0.5 s
    dt = d["timestamp_s"].diff().fillna(1)
    gap = (dt > 0.5) | (dt <= 0)                       # time gap, or the next session starting
    seg = (gap | (d["activity_id"] != d["activity_id"].shift())
           | (d["session"] != d["session"].shift())).cumsum()
    cols = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
    parts = []
    for _, s in d.groupby(seg):
        if len(s) < 2 * WIN:
            continue
        t = s["timestamp_s"].to_numpy()
        grid = np.arange(t[0], t[-1], 1 / FS_IN)            # even 100 Hz grid (fills tiny gaps)
        u = np.column_stack([np.interp(grid, t, s[c].to_numpy()) for c in cols])
        x = resample_poly(u, up=2, down=5, axis=0)          # 100 Hz -> 40 Hz (with anti-alias filter)
        parts.append((x, s["label"].iloc[0], s["activity"].iloc[0]))
    return int(d["subject"].iloc[0]), d["wrist_side"].iloc[0], parts

subjects, seen = [], set()
for f in files:
    s = load_subject(f)
    if s[0] in seen:                     # the same subject uploaded twice: use it only once
        print("skipped duplicate:", os.path.basename(f)); continue
    seen.add(s[0]); subjects.append(s)
for sid, side, parts in subjects:
    mins = pd.Series({lab: 0.0 for lab in CLASSES + ["test_only"]})
    for x, lab, _ in parts:
        mins[lab] += len(x) / FS_OUT / 60
    print(sid, side, "minutes:", mins.round(1).to_dict())

## 2. Windows and features
Every feature uses the **magnitude** of acceleration or rotation, so it doesn't matter how the
watch is rotated on the wrist, or which wrist (subject 108 is left-wristed).
These exact formulas must be re-implemented on the watch later.

In [ ]:
FEATURES = ["acc_mean", "acc_std", "acc_min", "acc_max", "acc_jerk",
            "gyro_mean", "gyro_std", "gyro_max", "cadence_hz", "cadence_power"]

def features(w):
    am = np.linalg.norm(w[:, :3], axis=1)        # acceleration magnitude (g)
    gm = np.linalg.norm(w[:, 3:], axis=1)        # rotation magnitude (deg/s)
    # cadence: strongest rhythm between 0.5 and 4 Hz in the acceleration magnitude
    spec = np.abs(np.fft.rfft(am - am.mean(), n=256)) ** 2
    freqs = np.fft.rfftfreq(256, d=1 / FS_OUT)
    band = (freqs >= 0.5) & (freqs <= 4.0)
    k = np.argmax(spec[band])
    total = spec.sum() + 1e-9
    return [am.mean(), am.std(), am.min(), am.max(), np.abs(np.diff(am)).mean(),
            gm.mean(), gm.std(), gm.max(),
            freqs[band][k], spec[band][k] / total]

rows, seg_no = [], 0
for sid, side, parts in subjects:
    for x, lab, act in parts:
        seg_no += 1                                         # one unbroken stretch of one activity
        for i in range(0, len(x) - WIN + 1, WIN):          # non-overlapping 1.5 s windows
            rows.append([sid, lab, act, seg_no] + features(x[i:i + WIN]))

allwin = pd.DataFrame(rows, columns=["subject", "label", "activity", "segment"] + FEATURES)
data = allwin[allwin["label"].isin(CLASSES)].reset_index(drop=True)      # used for training/testing
unseen = allwin[allwin["label"] == "test_only"].reset_index(drop=True)   # never trained on
print(len(data), "windows for the 3 classes,", len(unseen), "test-only windows")
print(pd.crosstab(data["subject"], data["label"])[CLASSES])
print("\nSedentary windows by activity:", data[data["label"] == "sedentary"]["activity"].value_counts().to_dict())

## 3. Quick look: do the features separate the classes?
Early evidence for claim C14 (magnitude and cadence separate the three activities).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, ["acc_std", "gyro_mean", "cadence_hz"]):
    data.boxplot(column=f, by="label", ax=ax, positions=None)
    ax.set_title(f); ax.set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.show()

## 4. Leave-one-subject-out test
Never split by window: windows from the same person would end up in both training and test,
and the error would look better than it really is.
`class_weight="balanced"` stops the model favouring sedentary just because there's more of it.

In [ ]:
MODELS = {
    "Random forest (10 trees, depth 6)": lambda: RandomForestClassifier(
        n_estimators=10, max_depth=6, class_weight="balanced", random_state=0),
    "Decision tree (depth 5)": lambda: DecisionTreeClassifier(
        max_depth=5, class_weight="balanced", random_state=0),
    "Logistic regression (baseline)": lambda: make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)),
}

X, y, groups = data[FEATURES].to_numpy(), data["label"].to_numpy(), data["subject"].to_numpy()
results, conf = {}, {}
for name, make in MODELS.items():
    per_subject, cm = {}, np.zeros((3, 3), int)
    for s in np.unique(groups):
        tr, te = groups != s, groups == s
        m = make().fit(X[tr], y[tr])
        pred = m.predict(X[te])
        per_subject[s] = 100 * np.mean(pred != y[te])
        cm += confusion_matrix(y[te], pred, labels=CLASSES)
    results[name], conf[name] = per_subject, cm

table = pd.DataFrame(results).round(1)
table.index.name = "left-out subject"
table.loc["MEAN"] = table.mean().round(1)
print("Error rate (%) on the left-out subject.  Target: mean <= 15 %")
print(table.to_string())

## 5. Which classes get confused? (all folds added together)

In [ ]:
fig, axes = plt.subplots(1, len(conf), figsize=(5 * len(conf), 4))
for ax, (name, cm) in zip(axes, conf.items()):
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

# Per-class recall for the forest (subjects 103, 104 and 109 have little or no running,
# so running is mostly tested on the other six)
cm = conf["Random forest (10 trees, depth 6)"]
for i, c in enumerate(CLASSES):
    print(f"{c:<10} recall {100 * cm[i, i] / max(cm[i].sum(), 1):5.1f} %   ({cm[i].sum()} windows)")

## 5b. Before and after: what did the extra sedentary data change?
Three versions of the sedentary class, same model, same leave-one-subject-out test:
1. protocol only (lying, sitting, standing), the first version of this notebook;
2. plus watching TV and computer work (adds subject 109);
3. plus car driving (sitting with arm movement and vibration).

In [ ]:
def loso_error(df):
    Xd, yd, gd = df[FEATURES].to_numpy(), df["label"].to_numpy(), df["subject"].to_numpy()
    errs, cm = {}, np.zeros((3, 3), int)
    for s in np.unique(gd):
        m = MODELS["Random forest (10 trees, depth 6)"]().fit(Xd[gd != s], yd[gd != s])
        p = m.predict(Xd[gd == s])
        errs[s] = 100 * np.mean(p != yd[gd == s])
        cm += confusion_matrix(yd[gd == s], p, labels=CLASSES)
    recall = {c: round(float(100 * cm[i, i] / max(cm[i].sum(), 1)), 1) for i, c in enumerate(CLASSES)}
    return np.mean(list(errs.values())), recall

V1 = data[~data["activity"].isin(["watching_tv", "computer_work", "car_driving"])]
V2 = data[data["activity"] != "car_driving"]
versions = [("1. protocol sedentary only", V1), ("2. + TV and computer work", V2),
            ("3. + car driving", data)]
for name, df in versions:
    e, r = loso_error(df)
    print(f"{name:<32} subjects={df['subject'].nunique()}  windows={len(df):5d}  mean error {e:4.1f} %   recall {r}")

## 5c. Activities the model was never trained on
Real days include stairs and chores. The model can only answer sedentary, walking or
running, so this shows which of those it picks. Each test-only activity is predicted by a model
trained **without that subject**, so nobody is tested on their own data.

In [ ]:
Xu = unseen[FEATURES].to_numpy()
pred_u = np.empty(len(unseen), dtype=object)
for s in np.unique(unseen["subject"]):
    m = MODELS["Random forest (10 trees, depth 6)"]().fit(X[groups != s], y[groups != s])
    pred_u[unseen["subject"].to_numpy() == s] = m.predict(Xu[unseen["subject"].to_numpy() == s])
unseen["pred"] = pred_u
tab = (pd.crosstab(unseen["activity"], unseen["pred"], normalize="index") * 100).round(0)
tab = tab.reindex(columns=CLASSES, fill_value=0)
tab["windows"] = unseen["activity"].value_counts()
print("Share of windows given each label (%):")
print(tab.to_string())

## 5d. Smoothing, exactly as the watch does it
The watch shows a new label only after **3 windows in a row** agree (4.5 s), so a brief arm
movement while sitting does not flip the label to walking. The first window after switching on
is shown straight away. Here the same rule is applied to the leave-one-subject-out predictions,
in time order, within each unbroken stretch of one activity.
Cost: a real change of activity shows up about 3 windows (4.5 s) later.

In [ ]:
STABLE_N = 3

def smooth(preds, n=STABLE_N):
    out, stable, last, run = [], None, None, 0
    for p in preds:
        run = run + 1 if p == last else 1
        last = p
        if stable is None or run >= n:
            stable = p
        out.append(stable)
    return out

raw_pred = np.empty(len(data), dtype=object)
for s in np.unique(groups):
    m = MODELS["Random forest (10 trees, depth 6)"]().fit(X[groups != s], y[groups != s])
    raw_pred[groups == s] = m.predict(X[groups == s])
data["raw_pred"] = raw_pred
data["smooth_pred"] = np.concatenate([smooth(list(g["raw_pred"])) for _, g in data.groupby("segment", sort=False)])
cmp_rows = []
for name, col in [("raw (each window on its own)", "raw_pred"), (f"smoothed ({STABLE_N} agreeing windows)", "smooth_pred")]:
    per_s = data.groupby("subject").apply(lambda g: 100 * np.mean(g[col] != g["label"]), include_groups=False)
    rec = {c: round(float(100 * np.mean(data.loc[data["label"] == c, col] == c)), 1) for c in CLASSES}
    flips = int(sum((g[col].values[1:] != g[col].values[:-1]).sum() for _, g in data.groupby("segment", sort=False)))
    cmp_rows.append({"predictions": name, "mean_error_pct": round(float(per_s.mean()), 2), **{f"recall_{c}_pct": v for c, v in rec.items()},
                     "label_flips_inside_one_activity": flips})
smoothing_table = pd.DataFrame(cmp_rows)
print(smoothing_table.to_string(index=False))

## 6. Final model on all 9 subjects, and what it relies on
This is the model you'd shrink and put on the watch (claim C6). The importance chart shows
which features matter, which is useful for the report.

In [ ]:
final = MODELS["Random forest (10 trees, depth 6)"]().fit(X, y)
imp = pd.Series(final.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 4), title="Feature importance (random forest)")
plt.tight_layout(); plt.show()
nodes = sum(t.tree_.node_count for t in final.estimators_)
print(f"{len(final.estimators_)} trees, {nodes} nodes in total (small enough for the ESP32)")

# Part 3: export the model for the watch
This turns the random forest from step 6 into plain C code (`activity_model.h`).
Put that file in the `ActivityClassifier` sketch folder, next to `ActivityClassifier.ino`,
and upload the sketch to the watch. The watch then classifies live, with no computer involved.

Re-run this cell whenever you retrain, and replace the file in the sketch folder.

In [ ]:
import zlib

def export_forest_to_c(model, feature_names, path="activity_model.h"):
    names = [str(c) for c in model.classes_]
    out = []
    out.append("// activity_model.h - generated by wrist_classifier_pamap2.ipynb. Do not edit by hand.")
    out.append(f"// Random forest: {len(model.estimators_)} trees, trained on PAMAP2 wrist data.")
    out.append("// Features (same order as activity_features.h):")
    for i, f in enumerate(feature_names):
        out.append(f"//   x[{i}] = {f}")
    out.append("#pragma once\n")
    out.append(f"#define ACT_N_CLASSES {len(names)}")
    out.append(f"#define ACT_N_FEATURES {len(feature_names)}")
    out.append("static const char *ACT_CLASS_NAMES[ACT_N_CLASSES] = {"
               + ", ".join(f'"{n}"' for n in names) + "};\n")

    def node_code(t, node, depth):
        pad = "  " * depth
        if t.children_left[node] == -1:                        # leaf: add class shares to the votes
            v = t.value[node][0]
            v = v / v.sum()
            return [pad + " ".join(f"p[{k}] += {v[k]:.6f}f;" for k in range(len(v)) if v[k] > 0)]
        f, thr = t.feature[node], t.threshold[node]
        lines = [pad + f"if (x[{f}] <= {thr:.17g}) {{"]       # same comparison as scikit-learn
        lines += node_code(t, t.children_left[node], depth + 1)
        lines.append(pad + "} else {")
        lines += node_code(t, t.children_right[node], depth + 1)
        lines.append(pad + "}")
        return lines

    for i, est in enumerate(model.estimators_):
        out.append(f"static inline void act_tree_{i}(const float *x, float *p) {{")
        out += node_code(est.tree_, 0, 1)
        out.append("}\n")

    out.append("// Returns the class index; prob (optional) gets the share of votes for each class.")
    out.append("static inline int act_predict(const float *x, float *prob) {")
    out.append("  float p[ACT_N_CLASSES] = {0};")
    for i in range(len(model.estimators_)):
        out.append(f"  act_tree_{i}(x, p);")
    out.append("  int best = 0;")
    out.append("  for (int k = 0; k < ACT_N_CLASSES; k++) {")
    out.append(f"    p[k] /= {len(model.estimators_)}.0f;")
    out.append("    if (p[k] > p[best]) best = k;")
    out.append("    if (prob) prob[k] = p[k];")
    out.append("  }")
    out.append("  return best;")
    out.append("}")
    # fingerprint of this exact model, so replay.bin can check it was made for the same model
    model_id = zlib.crc32("\n".join(out).encode())
    out.insert(out.index("#pragma once\n") + 1, f"#define ACT_MODEL_ID 0x{model_id:08X}u")
    with open(path, "w") as fh:
        fh.write("\n".join(out) + "\n")
    print(f"wrote {path}: {len(model.estimators_)} trees, classes {names}, model id 0x{model_id:08X}")
    return model_id

MODEL_ID = export_forest_to_c(final, FEATURES)
try:
    from google.colab import files       # in Colab: download the file to your computer
    files.download("activity_model.h")
except ImportError:
    pass

# Part 4: replay file for the watch's SD card
`replay.bin` holds every PAMAP2 window, already converted to what the watch uses (40 Hz, g, deg/s),
with its true label and **the prediction Colab made for it**.

Copy it to the root of the watch's SD card. In replay mode the watch runs each window through
its own copy of the model and reports:
* **Agreement with Colab**: should be 100 %. This proves the watch computes exactly what the notebook does.
* **Accuracy** against the true labels, and the **time per window** (evidence for claim C6).

The model was trained on these same 9 people, so this accuracy is *not* a test on unseen people.
The leave-one-subject-out table in step 4 is the honest error rate.
Export `activity_model.h` and `replay.bin` together, from the same run.

In [ ]:
import struct

def export_replay(path="replay.bin"):
    idx = {c: i for i, c in enumerate(final.classes_)}
    recs = []
    for sid, side, parts in subjects:
        for x, lab, act in parts:
            if lab not in idx:                  # test-only activities have no true label
                continue
            for i in range(0, len(x) - WIN + 1, WIN):
                recs.append((sid, idx[lab], x[i:i + WIN].astype("<f4")))
    feats = np.array([features(w.astype(np.float64)) for _, _, w in recs])
    colab = [idx[p] for p in final.predict(feats)]
    correct = sum(int(c == r[1]) for c, r in zip(colab, recs))
    with open(path, "wb") as fh:
        # header, 32 bytes: magic, version, windows, window length, classes, model id, Colab correct count, spare
        fh.write(struct.pack("<4s7I", b"ACTR", 1, len(recs), WIN, len(final.classes_), MODEL_ID, correct, 0))
        for (sid, lab, w), c in zip(recs, colab):
            fh.write(struct.pack("<4B", sid - 100, lab, c, 0))   # subject 1-9, true label, Colab prediction
            fh.write(w.tobytes())
    print(f"wrote {path}: {len(recs)} windows, {os.path.getsize(path) / 1e6:.1f} MB")
    print(f"Colab accuracy on these windows: {100 * correct / len(recs):.1f} %  (the watch should show the same)")

export_replay()
try:
    from google.colab import files
    files.download("replay.bin")
except ImportError:
    pass